**1. Conectar con Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/Proyecto_Final_IA'
print(os.listdir(BASE))
print(os.listdir(f'{BASE}/Dataset'))

Mounted at /content/drive
['Dataset', 'P1_Notas', 'Code']
['challenge_data_json', 'challenge_data', 'src_experiment_data', 'src_experiment_data_json']


**2. Obtener el listado de ficheros de las carpetas**

In [ ]:
import os, pandas as pd

BASE = '/content/drive/MyDrive/Proyecto_Final_IA'
DATA = f'{BASE}/Dataset'
OUT = f'{BASE}/P1_Notas'

for carpeta in ['challenge_data', 'src_experiment_data']:
    print(carpeta, '->', sorted(os.listdir(f'{DATA}/{carpeta}')))

challenge_data -> ['.DS_Store', 'clef_taskC_test3.csv', 'clef_taskC_test3.csv\uf03aZone.Identifier', 'clef_taskC_test3_metadata.csv', 'clef_taskC_test3_metadata.csv\uf03aZone.Identifier', 'clinicalnlp_taskB_test1.csv', 'clinicalnlp_taskB_test1.csv\uf03aZone.Identifier', 'clinicalnlp_taskB_test1_metadata.csv', 'clinicalnlp_taskB_test1_metadata.csv\uf03aZone.Identifier', 'clinicalnlp_taskC_test2.csv', 'clinicalnlp_taskC_test2.csv\uf03aZone.Identifier', 'clinicalnlp_taskC_test2_metadata.csv', 'clinicalnlp_taskC_test2_metadata.csv\uf03aZone.Identifier', 'train.csv', 'train.csv\uf03aZone.Identifier', 'train_metadata.csv', 'train_metadata.csv\uf03aZone.Identifier', 'valid.csv', 'valid.csv\uf03aZone.Identifier', 'valid_metadata.csv', 'valid_metadata.csv\uf03aZone.Identifier']
src_experiment_data -> ['test1_aci_asr.csv', 'test1_aci_asr.csv\uf03aZone.Identifier', 'test1_aci_asr_metadata.csv', 'test1_aci_asr_metadata.csv\uf03aZone.Identifier', 'test1_aci_asrcorr.csv', 'test1_aci_asrcorr.csv\uf03

**3. Ver la estructura de un fichero de ejemplo**

In [ ]:
df = pd.read_csv(f'{DATA}/challenge_data/train.csv')
print(df.shape)
print(df.columns.tolist())
print(df.iloc[0]['dialogue'][:500])
print(df.iloc[0]['note'][:500])

(67, 4)
['dataset', 'encounter_id', 'dialogue', 'note']
[doctor] hi , martha . how are you ?
[patient] i'm doing okay . how are you ?
[doctor] i'm doing okay . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you , okay ?
[patient] okay .
[doctor] martha is a 50-year-old female with a past medical history significant for congestive heart failure , depression and hypertension who presents for her annual exam . so , martha , it's been a year since i've seen you . how are you doing ?
[patient] i'm doing well . i've been
CHIEF COMPLAINT

Annual exam.

HISTORY OF PRESENT ILLNESS

Martha Collins is a 50-year-old female with a past medical history significant for congestive heart failure, depression, and hypertension who presents for her annual exam. It has been a year since I last saw the patient.

The patient has been traveling a lot recently since things have gotten a bit better. She reports that she got her COVID-19 vaccine so she feels safer about travel

**4. Comprobar los títulos más frecuentes**

In [ ]:
import collections
cab = collections.Counter()
for n in df['note']:
    for linea in n.split('\n'):
        linea = linea.strip()
        if linea and linea.isupper() and len(linea) < 60:
            cab[linea] += 1
print(cab.most_common(20))

[('CHIEF COMPLAINT', 59), ('RESULTS', 52), ('REVIEW OF SYSTEMS', 50), ('HISTORY OF PRESENT ILLNESS', 45), ('PHYSICAL EXAM', 44), ('ASSESSMENT AND PLAN', 34), ('INSTRUCTIONS', 32), ('PLAN', 32), ('ASSESSMENT', 29), ('SOCIAL HISTORY', 28), ('VITALS', 23), ('MEDICATIONS', 19), ('MEDICAL HISTORY', 18), ('PHYSICAL EXAMINATION', 16), ('FAMILY HISTORY', 10), ('PAST HISTORY', 9), ('CURRENT MEDICATIONS', 8), ('ALLERGIES', 7), ('SURGICAL HISTORY', 7), ('CC:', 4)]


**5. Cargar el modelo Qwen2.5-7B-Instruct en 4 bits**

In [ ]:
#!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.8 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = 'Qwen/Qwen2.5-7B-Instruct'
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto')
print(f'{torch.cuda.memory_allocated()/1e9:.1f} GB en GPU')

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

5.9 GB en GPU


**6. Generar notas**

In [ ]:
SYSTEM = """You are a medical scribe. Write a clinical note from a doctor-patient conversation.

Use these section headings, in uppercase, in this order:
CHIEF COMPLAINT
HISTORY OF PRESENT ILLNESS
REVIEW OF SYSTEMS
PHYSICAL EXAM
RESULTS
ASSESSMENT AND PLAN

Section rules:
- CHIEF COMPLAINT: the reason for the visit in a few words (e.g. "Annual exam."). Always include it.
- HISTORY OF PRESENT ILLNESS: start with age, sex and relevant past medical history, then the symptoms, habits, medication adherence and concerns the patient reports.
- REVIEW OF SYSTEMS: bullet list by system ("• System: finding."), only for symptoms the patient reports or denies in the conversation (symptoms only, never exam findings or test results). Do NOT list systems that were not discussed.
- PHYSICAL EXAM: only findings that the doctor explicitly states during the exam. Do not add normal findings or vital sign values that are not mentioned.
- RESULTS: only lab, imaging or test results (e.g. echocardiogram, lipid panel).
- ASSESSMENT AND PLAN: one short assessment sentence, then the plan for each problem.

General rules:
- Third person, professional clinical language.
- Use only information stated in the conversation. Never invent findings or add phrases like "no other abnormalities".
- Never fill in typical or normal findings. If it was not said in the conversation, do not write it.
- Copy specific details exactly as stated (grades, doses, numbers, frequencies). Do not generalize them.
- The doctor may mention symptoms from an intake form (e.g. "you were endorsing nasal congestion"). Include them in the HPI and REVIEW OF SYSTEMS.
- Include every test or follow-up the doctor orders or plans to repeat in ASSESSMENT AND PLAN.
- If a section has no information (except CHIEF COMPLAINT), omit it.
- Do not add any text before or after the note."""

def generar_nota(dialogo, max_tokens=1024):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Conversation:\n\n{dialogo}\n\nWrite the clinical note."},
    ]
    texto = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(texto, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [ ]:
for i in [0, 1, 2]:
    print(f'===== CONSULTA {i} =====')
    print(generar_nota(df.iloc[i]['dialogue']))
    print()

===== CONSULTA 0 =====
CHIEF COMPLAINT
Annual exam

HISTORY OF PRESENT ILLNESS
50-year-old female with a past medical history significant for congestive heart failure, depression, and hypertension. Reports traveling frequently, hiking, and getting vaccinated. Mentions taking medications regularly and no recent symptoms of chest pain, shortness of breath, or leg swelling. Reports attending weekly therapy sessions for depression. Mentions difficulty remembering to take blood pressure medications and noticing increased blood pressure during stressful periods at work. No recent symptoms of nausea, vomiting, or abdominal pain.

REVIEW OF SYSTEMS
• Cardiovascular: Three out of six systolic ejection murmur noted.
• Respiratory: No symptoms reported.
• Gastrointestinal: No nausea, vomiting, or abdominal pain reported.
• Musculoskeletal: One plus pitting edema noted in lower extremities.

PHYSICAL EXAM
Blood pressure elevated.
Heart exam: Three out of six systolic ejection murmur.
Lower extremi

**7. Generar las notas de todo**

In [ ]:
import os, time
import pandas as pd

SALIDA = f'{OUT}/zero_shot_v1'
os.makedirs(SALIDA, exist_ok=True)

nombre = 'train'
data = pd.read_csv(f'{DATA}/challenge_data/{nombre}.csv')
# data = data.head(20)   # descomenta para probar solo con 20

destino = f'{SALIDA}/{nombre}_notas.csv'

if os.path.exists(destino):
    hechas = pd.read_csv(destino)
else:
    hechas = pd.DataFrame(columns=['encounter_id', 'nota_generada', 'nota_referencia'])
ids_hechos = set(hechas['encounter_id'])

filas = hechas.to_dict('records')
t0 = time.time()
for _, fila in data.iterrows():
    if fila['encounter_id'] in ids_hechos:
        continue
    filas.append({
        'encounter_id': fila['encounter_id'],
        'nota_generada': generar_nota(fila['dialogue'], max_tokens=1200),
        'nota_referencia': fila['note'],
    })
    if len(filas) % 5 == 0:
        pd.DataFrame(filas).to_csv(destino, index=False)
        print(f'{len(filas)}/{len(data)} ({(time.time()-t0)/60:.1f} min)')

pd.DataFrame(filas).to_csv(destino, index=False)
print(f'Terminado: {len(filas)} notas en {destino}')

5/67 (2.9 min)
10/67 (5.9 min)
15/67 (8.5 min)
20/67 (10.7 min)
25/67 (14.9 min)
30/67 (19.0 min)
35/67 (22.4 min)
40/67 (25.6 min)
45/67 (28.7 min)
50/67 (32.3 min)
55/67 (35.2 min)
60/67 (38.6 min)
65/67 (42.0 min)
Terminado: 67 notas en /content/drive/MyDrive/Proyecto_Final_IA/P1_Notas/zero_shot_v1/train_notas.csv
